**Policy RAG | Step 2: Embedding and retrieval**

Goal: turn the 33 clauses into searchable vectors and test the `search` function on a few questions.

The first run downloads the embedding model. Later runs use the saved copy.

**Setup.** Imports the project code in `src/policy_rag.py`.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd

# Find the project root: the folder that contains src/policy_rag.py
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "policy_rag.py").exists())
sys.path.append(str(ROOT / "src"))
import policy_rag as rag

POLICY_DIR = ROOT / "notebooks" / "RAG" / "policy"
OUT_DIR = ROOT / "notebooks" / "RAG" / "outputs"
OUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)

**Step 1. Chunk and build the index.** Clauses are embedded locally. BM25 (keyword search) is built on the same text.

In [2]:
chunks = rag.chunk_policy(rag.load_policy(POLICY_DIR / "marketplace_policy.txt"))

# Builds embeddings for all clauses. The clause text, with its section title, is what gets embedded.
index = rag.PolicyIndex(chunks)
print("clauses:", len(chunks), "| embedding matrix:", index.emb.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

clauses: 33 | embedding matrix: (33, 384)


**What is bge-small-en-v1.5, and why was something downloaded?**

An **embedding model** is a small neural network. You give it text, and it gives back a list of numbers (a vector). It was trained so that texts with a similar meaning get similar numbers, even if the words differ. That is how "How long does a Pix refund take?" can match clause 4.2.

The name breaks down like this:
- **BGE** means BAAI General Embedding, from the Beijing Academy of Artificial Intelligence.
- **small** is the size: about 33 million parameters, 384 numbers per text, about 130 MB. It runs on a normal CPU. The "base" and "large" versions are bigger and slower.
- **en** means English only. The policy is English, so this fits.
- **v1.5** is the version.

**Why sentence-transformers?** There are two separate things:
- `sentence-transformers` is a **Python library** (the tool). It was installed with `pip`. It gives you one simple call: `model.encode(texts)`.
- `bge-small-en-v1.5` is the **model** (the brain, a file of learned weights). The library downloaded it from the Hugging Face Hub on the first run. That is the "unauthenticated requests to HF Hub" warning and the `Loading weights` bar you saw.

The model is saved on your computer, so later runs do not download it again. We use a local model because it is free, and no policy text leaves your machine.

**"Normalized" and "dot product = cosine similarity"**

Think of each vector as an arrow. Cosine similarity measures how much two arrows point in the same direction:
- 1 means the same direction (very similar meaning)
- 0 means unrelated
- −1 means opposite

**Normalizing** means scaling the arrow so its length is exactly 1. Example: the vector `[3, 4]` has length 5. Divide each number by 5 and you get `[0.6, 0.8]`, which has length 1.

When both arrows have length 1, cosine similarity is just the **dot product**: multiply the numbers position by position, then add.

A toy example with 3 numbers (real vectors have 384, and the individual numbers mean nothing to a human):
- question: `[0.8, 0.6, 0.0]`
- clause 4.2: `[0.6, 0.8, 0.0]`, so the dot product is 0.8×0.6 + 0.6×0.8 + 0 = **0.96** (very close)
- clause 8.3: `[0.0, 0.6, 0.8]`, so the dot product is 0 + 0.36 + 0 = **0.36** (far)

The dot product is fast, so scoring all 33 clauses is one line: `self.emb @ q`.

**What does `(33, 384)` mean?**

It is the shape of a table of numbers (a matrix):
- **33 rows**: one per clause, in the same order as the `chunks` list. Row 0 is MOP-1.1, row 1 is MOP-1.2, and so on.
- **384 columns**: the 384 numbers that describe each clause.

So `index.emb` holds 33 × 384 = 12,672 decimal numbers. Each row is the "meaning fingerprint" of one clause.

**Step 2. Try the three search modes.** `dense` = search by meaning, `bm25` = search by words, `hybrid` = both combined.

In [3]:
queries = [
    "Which tier is a seller with 96% on-time delivery and 11% adverse reviews?",
    "How long does a Pix refund take?",
    "Who pays freight when a customer changes their mind?",
]
rows = []
for q in queries:
    for mode in ["dense", "bm25", "hybrid"]:
        hits = index.search(q, k=3, mode=mode)
        rows.append({"question": q[:45], "mode": mode,
                     "top3": ", ".join(h["chunk_id"] for h in hits),
                     "top1_cosine": round(hits[0]["dense_score"], 3)})
pd.DataFrame(rows)

,question,mode,top3,top1_cosine
0,Which tier is a seller with 96% on-time deliv,dense,"MOP-1.2, MOP-1.3, MOP-1.1",0.721
1,Which tier is a seller with 96% on-time deliv,bm25,"MOP-1.3, MOP-1.2, MOP-1.1",0.694
2,Which tier is a seller with 96% on-time deliv,hybrid,"MOP-1.2, MOP-1.3, MOP-1.1",0.721
3,How long does a Pix refund take?,dense,"MOP-4.2, MOP-7.1, MOP-6.5",0.825
4,How long does a Pix refund take?,bm25,"MOP-4.2, MOP-1.4, MOP-3.4",0.825
5,How long does a Pix refund take?,hybrid,"MOP-4.2, MOP-4.3, MOP-4.1",0.825
6,Who pays freight when a customer changes thei,dense,"MOP-8.3, MOP-8.4, MOP-8.1",0.700
7,Who pays freight when a customer changes thei,bm25,"MOP-8.3, MOP-7.3, MOP-6.2",0.700
8,Who pays freight when a customer changes thei,hybrid,"MOP-8.3, MOP-8.4, MOP-8.1",0.700


**What is `1/(60+rank)` (reciprocal rank fusion)?**

Hybrid search has two ranked lists, one from dense and one from BM25. The problem is that their scores are on different scales (cosine is about 0.7, BM25 is about 10 or 12), so they cannot simply be added. RRF ignores the scores and uses only the **rank** (1st, 2nd, 3rd...) in each list. Each clause gets `1/(60 + rank)` from each list, and the two parts are added.

An example with made-up ranks:
- Clause X is rank 1 in dense and rank 1 in BM25: 1/61 + 1/61 = **0.0328**
- Clause Y is rank 2 in dense and rank 30 in BM25: 1/62 + 1/90 = **0.0272**
- Clause Z is rank 4 in dense and rank 3 in BM25: 1/64 + 1/63 = **0.0315**

Final order: X, then Z, then Y. Z is only medium in both lists but still beats Y, which is strong in one list and weak in the other. 

The 60 is a standard smoothing constant. It stops rank 1 from beating rank 2 by too much.

**How are the embeddings stored?**

- They are stored **only in memory**, in `index.emb`, a numpy array of 33 rows and 384 columns. Nothing is saved to disk. For 33 clauses it takes seconds to rebuild each time.
- Row i matches `chunks[i]` **by position**. That is how a score at position 5 turns back into a clause ID.
- A search is four steps: embed the question into 384 numbers, multiply with the matrix to get 33 scores, sort the scores, and take the top k clause IDs.
- A vector database (FAISS, Chroma) is only needed for millions of chunks. For 33, numpy is enough.

**Step 3. Look at one result in full.** Shows what `search` returns for one question.

In [4]:
for h in index.search("Which tier is a seller with 96% on-time delivery and 11% adverse reviews?", k=3):
    print(h["rank"], h["chunk_id"], "| cosine:", round(h["dense_score"], 3), "| bm25:", round(h["bm25_score"], 2))
    print("   ", h["text"][:110].replace("\n", " "))

1 MOP-1.2 | cosine: 0.721 | bm25: 10.16
    1.2 Tiers are determined by two measures: on-time delivery rate (OTD) and adverse review rate (ARR, the share 
2 MOP-1.3 | cosine: 0.694 | bm25: 12.51
    1.3 A seller must satisfy both thresholds for a tier. A seller with 96% OTD and 11% ARR is Bronze, not Gold.
3 MOP-1.1 | cosine: 0.652 | bm25: 6.24
    1.1 Every active seller is assigned a performance tier each calendar month, calculated on the trailing 90 days


**Additional Code for Reference**

In [5]:
print(index.emb.shape)

(33, 384)


In [6]:
print(index.emb[0][:10])    # first 10 of the 384 numbers of MOP-1.1

[-0.00108163 -0.04151946  0.01100057 -0.00270585 -0.03407948 -0.02156831
  0.08203968  0.0119303   0.07231343  0.07714771]


In [7]:
# this is the query embedding for "How long does a Pix refund take?"
q = index.embedder.encode([rag.QUERY_PREFIX + "How long does a Pix refund take?"],normalize_embeddings=True)[0]

# compute cosine similarities between the query and all clause embeddings
scores = index.emb @ q  

scores      # 33 scores, one per clause

array([0.51557124, 0.48725012, 0.47877556, 0.5740795 , 0.49353257,
       0.5046512 , 0.59462416, 0.59608537, 0.54696596, 0.60155445,
       0.5351251 , 0.5007577 , 0.53529394, 0.57945585, 0.6099911 ,
       0.8252748 , 0.615614  , 0.4575391 , 0.52917707, 0.526245  ,
       0.6077938 , 0.5362615 , 0.4729594 , 0.6003477 , 0.58092755,
       0.64988506, 0.666453  , 0.53451747, 0.59680057, 0.47075522,
       0.44068122, 0.52605116, 0.5563092 ], dtype=float32)

In [8]:
# top 3 clauses with the highest cosine similarity to the query
top = scores.argsort()[::-1][:3]
print([(chunks[i]["chunk_id"], round(float(scores[i]), 3)) for i in top])

[('MOP-4.2', 0.825), ('MOP-7.1', 0.666), ('MOP-6.5', 0.65)]
